# Этап 12 V1 — Research Synthesis Stage 1–11

## Исследовательский вопрос

Что Stage 1–11 действительно показали о качестве модели без `Q_B1_norm` / `Q_B2_norm`, основном ограничении текущего решения и ожидаемом information gain ещё одной architecture на тех же 47 признаках?

Это синтез уже accepted evidence. Он не меняет feature contract, split, seed или протокол Stage 1–11; не открывает final test и не создаёт новых model results.

## Чтение и проверка источников

Ниже читаются только Stage 12 evidence JSON и указанные в нём accepted summaries/results. Проверка подтверждает dataset/protocol identity, наличие источников и запрет использования final test.

In [1]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent

evidence_path = ROOT / 'reports/generated/stage12_research_synthesis_evidence_V1.json'
evidence = json.loads(evidence_path.read_text(encoding='utf-8'))
assert evidence['synthesis_only'] is True
assert evidence['final_test_used'] is False
assert evidence['protocol_identity']['features_allowed'] == 47
assert evidence['protocol_identity']['seed'] == 42
assert evidence['protocol_identity']['cv'] == '3-fold StratifiedKFold'

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

for item in evidence['source_artifacts']:
    source_path = ROOT / item['path']
    assert source_path.exists(), source_path
    assert sha256(source_path) == item['sha256'], source_path

assert len(evidence['source_notebooks']) == 11
for notebook_path in evidence['source_notebooks']:
    accepted_notebook = ROOT / notebook_path
    assert accepted_notebook.exists(), accepted_notebook

print('Проверено accepted artifacts:', len(evidence['source_artifacts']))
print('Проверено accepted notebooks:', len(evidence['source_notebooks']))
print('Final test использован:', evidence['final_test_used'])

Проверено accepted artifacts: 22
Проверено accepted notebooks: 11
Final test использован: False


## Что проверено

- Stage 4 использует accepted V2, Stage 6 — accepted V4; intermediate/rejected версии не включены.
- Каждая численная строка итоговой таблицы содержит путь к concrete accepted summary.
- `final_test_used = false` закреплён и в evidence, и в summary Stage 12.

## Общая таблица Stage 1–11

Этот шаг читает готовую CSV-таблицу, собранную только из зафиксированных summaries. Метрики не пересчитываются и не смешиваются: Gini, Recall@0.5, capacity-rescue и oracle-union остаются отдельными видами evidence.

In [2]:
import csv

table_path = ROOT / 'reports/generated/stage12_research_synthesis_table_V1.csv'
with table_path.open(encoding='utf-8', newline='') as handle:
    table = list(csv.DictReader(handle, delimiter=';'))

assert len(table) == 11
assert [row['Stage'] for row in table] == [f'Stage {n} V{v}' for n, v in [(1, 2), (2, 1), (3, 1), (4, 2), (5, 1), (6, 4), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1)]]
assert all(row['Primary evidence'].startswith('reports/summary/') for row in table)

for row in table:
    print(f"{row['Stage']}: {row['Decision']}")

Stage 1 V2: Принять baseline Gini ≈0.804
Stage 2 V1: Принять устойчивое consensus-ядро
Stage 3 V1: Перейти к diagnostic закрытых сигналов
Stage 4 V2: Stage 4 V2 accepted
Stage 5 V1: material_missing_signal
Stage 6 V4: inferior
Stage 7 V1: no_material_benefit
Stage 8 V1: no_material_benefit
Stage 9 V1: no_material_rank_complementarity
Stage 10 V1: limited_residual_model_reserve
Stage 11 V1: inferior


# Что показали Stage 1–11

## ФАКТЫ

### 1. Сильный baseline без закрытых индексов существует

На одних и тех же 47 разрешённых признаках три GBDT дали близкое качество:

- XGBoost: `Gini = 0.8040`;
- CatBoost: `Gini = 0.8038`;
- LightGBM: `Gini = 0.8034`.

Позднее сохранённый `GBDT_mean` дал:

**`Gini = 0.806399`.**

Это закрывает первый важный вопрос исследования положительно:

**сильную воспроизводимую рабочую основу можно построить без использования `Q_B1_norm` и `Q_B2_norm` как predictors.**

### 2. Основные тяжёлые ошибки не принадлежат одной конкретной GBDT

На рабочей выборке было `28 015` дефолтов.

Stage 3 выделил:

- `1 278` глубоко пропущенных дефолтов;
- из них `805` образуют общую blind spot;
- сильное disagreement между GBDT среди deeply missed случаев составляет только `1.9%`.

То есть значительная часть самых трудных ошибок повторяется у разных GBDT.

Это делает объяснение вида «неудачно выбрали конкретную boosting-модель» менее убедительным.

### 3. Внутри blind spot обнаружен диагностический information gap

Stage 4 показал, что закрытый показатель `Q_B2` заметно лучше различает объекты внутри общей blind spot:

- blind-spot ROC-AUC `Q_B2 = 0.723382`;
- для `Q_B1` — `0.697340`.

Но `Q_B2` используется только как diagnostic/reference signal и не является допустимым predictor рабочей модели.

Поэтому Stage 5 проверил другой вопрос:

**можно ли восстановить полезную часть сигнала `Q_B2` из уже разрешённых 47 признаков?**

Результат:

- oracle `Q_B2` blind AUC: `0.723382`;
- OOF proxy из 47 разрешённых признаков: `0.398056`;
- разница: `0.325326`.

Decision Stage 5:

**`material_missing_signal`.**

Разрешённые признаки воспроизводят часть общего сигнала, связанного с `Q_B2`, но не воспроизводят существенную часть сигнала, полезного именно для трудной blind spot.

## Что показал поиск альтернативных моделей

После обнаружения information gap мы отдельно проверили, можно ли получить существенный резерв **без изменения 47-feature representation**, только за счёт другого model pipeline.

### TabM standalone

Stage 6:

- `Gini = 0.781310`;
- `ΔGini = -0.022680` относительно control.

Decision:

**`inferior`.**

Самостоятельная TabM не улучшила сильный GBDT baseline.

### TabM stacking

Stage 7 проверил, появляется ли дополнительная ценность, если использовать TabM не вместо GBDT, а поверх уже имеющегося GBDT evidence.

Результат:

- `GBDT_mean Gini = 0.806399`;
- hybrid `Gini = 0.804260`;
- hybrid ниже baseline на `3 из 3` folds.

Decision:

**`no_material_benefit`.**

То есть stacking также не подтвердил существенного дополнительного резерва.

### FT-Transformer

Stage 8:

- FT-Transformer `Gini = 0.801528`;
- `ΔGini = -0.004871`.

Decision:

**`no_material_benefit`.**

При threshold `0.5` FT показывал более высокий Recall, поэтому этот результат потребовал отдельной проверки: действительно ли модель находит другие трудные дефолты или различие связано с выбранным числовым cutoff.

### Rank complementarity FT-Transformer

Stage 9 убрал зависимость от общего threshold и сравнил модели при одинаковой risk capacity.

При capacity `30%`:

- `GBDT_mean` rescue общей blind spot: `0 / 805`;
- FT-Transformer: `9 / 805`;
- преимущество FT: только `+1.118 п.п.`.

Decision:

**`no_material_rank_complementarity`.**

Небольшое отличие ranking существует, но заранее установленного material threshold оно не достигло.

### Oracle residual model reserve

Stage 10 задал ещё более благоприятный вопрос:

**если объединить правильные high-risk попадания всех уже сохранённых rankings, сколько blind spot вообще можно дополнительно спасти?**

При nominal capacity `30%`:

- oracle-any спас `15 из 805`;
- прирост — `+1.863 п.п.`;
- фактическая union capacity при этом выросла до `37.446%`.

Decision:

**`limited_residual_model_reserve`.**

Даже такой оптимистичный diagnostic upper bound не обнаружил большого скрытого model reserve.

### Независимый RealMLP

Stage 11 дал ещё одну независимую проверку современной model architecture:

- RealMLP `Gini = 0.793325`;
- `GBDT_mean = 0.806399`;
- `ΔGini = -0.013074`;
- RealMLP проиграл по Gini на всех `3 из 3` folds.

Decision:

**`inferior`.**

Таким образом, отрицательный результат уже не относится к одной конкретной neural architecture: разные model-level гипотезы последовательно не подтвердили material gain на неизменных 47 признаках.

## Фигуры синтеза

Шесть SVG-фигур ниже созданы только по saved accepted metrics. Этот код проверяет их комплектность; сами figure files уже входят в Stage 12 evidence package.

In [3]:
figure_dir = ROOT / 'reports/figures/stage12_research_synthesis_V1'
figures = sorted(figure_dir.glob('*.svg'))
assert len(figures) == 6
for figure in figures:
    assert figure.read_text(encoding='utf-8').lstrip().startswith('<svg')
    print(figure.name)

01_карта_цепочки.svg
02_oof_gini_моделей.svg
03_blind_spot.svg
04_qb2_oracle_proxy.svg
05_residual_rescue.svg
06_выводы_и_reopen.svg


# ИНТЕРПРЕТАЦИЯ

Stage 1–11 складываются в достаточно последовательную картину.

Сначала мы получили сильный baseline без закрытых `Q_B1_norm` и `Q_B2_norm`.

Затем выяснилось, что значительная часть тяжёлых ошибок повторяется сразу у нескольких сильных GBDT и образует общую blind spot.

После этого диагностические эксперименты с `Q_B2` показали, что внутри этой blind spot существует полезный сигнал, который текущие 47 разрешённых признаков воспроизводят недостаточно хорошо.

Наконец, несколько существенно разных попыток извлечь дополнительное качество **только сменой model architecture** не дали material gain:

- TabM standalone;
- TabM stacking;
- FT-Transformer;
- rank-complementarity FT;
- oracle residual reserve;
- RealMLP.

Поэтому совокупный evidence сильнее поддерживает следующую интерпретацию:

> **На текущем наборе из 47 разрешённых признаков основное ограничение скорее находится в доступной информации, чем в недостатке уже проверенных model architectures.**

Это важное различие.

Вывод не состоит в том, что «лучшей модели не существует» или что `Gini = 0.806399` является математическим потолком.

Вывод намного уже:

**на текущем evidence ожидаемая исследовательская ценность ещё одной model-only architecture стала низкой.**

Новая модель может случайно дать небольшое изменение метрик, но сейчас нет подтверждённой гипотезы, почему именно ещё одна архитектура должна устранить обнаруженную общую blind spot или воспроизвести недостающий диагностический сигнал.

Поэтому checkpoint

`CORE_MODEL_RESEARCH_STOPPED_CURRENT_47_FEATURES`

означает не завершение всего исследования, а завершение **поиска очередной архитектуры при неизменном 47-feature contract**.

# ОГРАНИЧЕНИЯ

Полученная цепочка evidence достаточно сильна для остановки текущего model-only search, но имеет важные границы.

### Temporal stability не проверена

Надёжной row-level observation date в текущем dataset нет.

Поэтому random CV / OOF показывает воспроизводимость результата на текущей выборке, но **не доказывает устойчивость модели во времени, отсутствие drift или качество на будущем временном периоде**.

### Information limitation не является математическим ceiling

Stage 1–11 не доказывают, что на 47 признаках невозможно получить Gini немного выше текущего результата.

Мы проверили конечный набор моделей и конкретных зафиксированных конфигураций.

Корректный вывод — низкий ожидаемый information gain следующего model-only эксперимента, а не доказанная невозможность улучшения.

### Missing signal не раскрывает конкретный новый признак

Stage 5 показывает, что разрешённые признаки недостаточно воспроизводят полезную часть diagnostic signal `Q_B2`.

Но experiment не отвечает на вопрос, **какой именно новый production-доступный признак способен этот разрыв закрыть**.

Также он не доказывает причинное влияние `Q_B2` на дефолт.

### Oracle diagnostics не являются реальной моделью

Stage 10 использует oracle union как оптимистичную диагностическую верхнюю границу model complementarity.

Это не deployable ensemble и не честное сравнение двух систем при одинаковой общей capacity.

### Threshold и business policy исследованы отдельно не были

Recall около `69%` остаётся business reference, а не автоматически оптимальной целью.

Без подтверждённых:

- review capacity;
- допустимого FP burden;
- стоимости FN/FP;
- другого formal operating constraint

нельзя утверждать, какой threshold или какой баланс Recall / Precision является оптимальным для Комус.

### Explainability не является причинностью

SHAP, permutation importance, ranking и diagnostic closed signals показывают модельные ассоциации и структуру ошибок.

Они не доказывают причин возникновения дефолта.

### Final test не использовался

Final test не применялся для выбора модели, признаков, hyperparameters, threshold или направления Stage 1–12.

`Q_B1_norm` и `Q_B2_norm` остаются только diagnostic/reference signals и не входят в разрешённый feature contract рабочей модели.

# Что закрыто и что остаётся открыто

## Что можно считать закрытым

На текущем evidence закрыт вопрос:

**нужно ли продолжать перебор model architectures на неизменных 47 разрешённых признаках?**

Ответ Stage 1–11:

**сейчас достаточного исследовательского основания для этого нет.**

Также подтверждено:

- сильный baseline без `Q_B1_norm/Q_B2_norm` существует;
- большая часть наиболее тяжёлых ошибок не является проблемой одной GBDT;
- внутри общей blind spot есть evidence недостающего информационного сигнала;
- проверенные современные model-only альтернативы не дали material gain;
- stacking не подтвердил существенный резерв;
- повышенный Recall FT при `0.5` не подтвердился как material rank-complementarity;
- среди уже сохранённых моделей не обнаружен большой residual model reserve;
- независимый RealMLP не дал основания снова открывать model search.

## Что остаётся открытым

Главный следующий содержательный вопрос находится уже на уровне информации:

> **Какие новые реально доступные и разрешённые данные способны добавить сигнал для общей blind spot, которого недостаточно в текущих 47 features?**

Но такой experiment нельзя открывать абстрактно.

Сначала должен появиться конкретный источник данных с понятным provenance и корректным способом привязки к объектам исследования.

Отдельно остаётся открытым business operating point:

- какую долю компаний допустимо отправлять в high-risk / review;
- какой FP burden приемлем;
- какое соотношение Recall и Precision действительно нужно бизнесу.

## Что сейчас заблокировано

Temporal validation и исследования drift остаются заблокированными отсутствием надёжной row-level observation date.

По той же причине современные внешние snapshots нельзя без дополнительного temporal evidence прикреплять к историческим строкам и трактовать как признаки, существовавшие на момент исторического решения.

Новая model architecture сама по себе эти ограничения не снимает.

# Когда model research стоит открыть снова

Checkpoint

`CORE_MODEL_RESEARCH_STOPPED_CURRENT_47_FEATURES`

не является необратимым запретом на новые модели.

Model research имеет смысл открыть снова, если появится **новое evidence, меняющее текущую постановку задачи**.

Достаточным основанием может стать:

1. **Новый валидный feature source**  
   Появляются новые разрешённые данные с понятным происхождением и корректной привязкой к наблюдениям.

2. **Надёжный temporal anchor**  
   Появляется row-level observation date или эквивалентная временная информация, позволяющая корректно проводить temporal validation или historical enrichment.

3. **Сильный независимый противоречащий результат**  
   Новый воспроизводимый experiment показывает material gain и тем самым противоречит текущему evidence о низком model reserve.

4. **Изменение формальной бизнес-задачи**  
   Заказчик фиксирует operating capacity, FN/FP policy или другой objective, из-за которого возникает новый исследовательский вопрос.

5. **Новая конкретная научная гипотеза**  
   До запуска можно ясно сформулировать, почему проверяемый механизм должен работать и как положительный либо отрицательный результат изменит решение проекта.

Само появление TabNet, NODE, SAINT, GRANDE, нового Transformer или ещё одной ML-библиотеки таким основанием не является.

# ИТОГОВЫЙ ВЫВОД

Stage 1–11 дали не просто таблицу результатов разных моделей, а последовательную исследовательскую цепочку:

**сильный 47-feature baseline  
→ общая структура тяжёлых ошибок  
→ локализация blind spot  
→ evidence недостающего диагностического сигнала  
→ проверка современных model-only альтернатив  
→ stacking  
→ rank-complementarity  
→ oracle residual reserve  
→ независимая проверка RealMLP.**

На текущем evidence наиболее поддержанная интерпретация следующая:

> **Основное ограничение текущего решения скорее связано с информацией, доступной в 47 разрешённых признаках, чем с недостатком уже проверенных model architectures.**

Поэтому зафиксирован исследовательский checkpoint:

**`CORE_MODEL_RESEARCH_STOPPED_CURRENT_47_FEATURES`**

Он означает:

**не открывать очередной model-only experiment на том же feature contract без новой содержательной гипотезы или нового evidence.**

Следующий практический этап проекта — использовать накопленные результаты как единый evidence package:

**research synthesis → итоговые figures → presentation / defence → затем необходимые reusable foundations.**

Исследование может быть содержательно продолжено при появлении нового корректного источника информации, temporal anchor, нового business objective или другого evidence, которое действительно меняет текущую постановку.

Stage 12 не использует final test и не создаёт новых model results: он синтезирует уже сохранённое accepted evidence Stage 1–11.

**Итоговый статус Stage 12 V1: `completed_accepted`.**

# СЛЕДУЮЩИЙ ШАГ

Stage 1–11 и текущий synthesis показывают, что дальнейший перебор model architectures на неизменных 47 разрешённых признаках сейчас имеет низкий ожидаемый information gain.

Поэтому следующий содержательный исследовательский вопрос находится уже не на уровне модели, а на уровне доступной информации:

> **Появляется ли дополнительный диагностический сигнал для общей blind spot при использовании новых данных или признаков, которых нет в текущих 47 разрешённых признаках?**

Этот вопрос следует открывать как новый experiment только после появления конкретного источника данных с понятным provenance и корректной привязкой к объектам исследования.

Без надёжной row-level observation date внешние данные могут использоваться только для диагностики потенциального missing signal. Такой анализ не будет доказывать temporal stability, historical uplift или пригодность современных snapshots как исторических признаков.

До появления такого источника данных:

**core model research на текущих 47 признаках остаётся остановленным, а Stage 12 является итоговым synthesis принятого evidence Stage 1–11.**